# Multi-sample variants

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmdcolin/jbrowse-anywidget/blob/main/examples/04_multisample_variants.ipynb)

A multi-sample VCF has one genotype column per sample. A `VariantTrack` can render it as a per-sample band or as a genotype matrix — that's the display `type` — and a samples TSV lets you group and color samples by metadata.

In [ ]:
# Install only if not already available (e.g. in Colab). The GitHub install
# needs no JS toolchain — the built widget bundle is committed in the repo. A
# local editable install is used as-is. (Swap to `jbrowse-anywidget` once it's
# published to PyPI.)
try:
    import jbrowse_anywidget  # noqa: F401
except ImportError:
    %pip install -q "jbrowse-anywidget @ git+https://github.com/cmdcolin/jbrowse-anywidget" pandas numpy

# Colab requires this to render third-party (anywidget) widgets:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

## A per-sample band, colored by population

`samplesTsvLocation` maps each sample to attributes (here `population`); the display's `colorBy` names the column that colors the rows. The VCF's `.tbi` index is resolved from the `uri`.

In [ ]:
from jbrowse_anywidget import LinearGenomeView, make_assembly

volvox = make_assembly(
    "volvox",
    "https://jbrowse.org/genomes/volvox/volvox.fa.gz",
)

base = (
    "https://raw.githubusercontent.com/GMOD/jbrowse-components/main/"
    "test_data/volvox/"
)

def sv_track(track_id, name, display_type):
    return {
        "type": "VariantTrack",
        "trackId": track_id,
        "name": name,
        "assemblyNames": ["volvox"],
        "adapter": {
            "type": "VcfTabixAdapter",
            "uri": base + "volvox.sv.vcf.gz",
            "samplesTsvLocation": {"uri": base + "volvox.sv.samples.tsv"},
        },
        "displays": [
            {
                "type": display_type,
                "displayId": track_id + "-display",
                "colorBy": "population",
            }
        ],
    }

view = LinearGenomeView(assembly=volvox, location="ctgA:1..50,000")
view.add_track(
    sv_track("sv-band", "multi-sample SV", "LinearMultiSampleVariantDisplay")
)
view

## The same VCF as a genotype matrix

Swap the display `type` to `LinearMultiSampleVariantMatrixDisplay` for a compact grid — one column per variant, one row per sample — that scales to hundreds of samples.

In [ ]:
matrix = LinearGenomeView(assembly=volvox, location="ctgA:1..50,000")
matrix.add_track(
    sv_track(
        "sv-matrix", "genotype matrix",
        "LinearMultiSampleVariantMatrixDisplay",
    )
)
matrix